In [ ]:
#put this script in the same folder with all the below files (ppid refers to participant id):
# 1. Minute physiological data
# 2. Minute posture data (Majoriites per 60sec_ppid_day1 or Majoriites per 60sec_ppid_day2)
# 3. EMA data (though this is optional as it is not provided on GitHub due to being sensitive data)
# 4. Lab calibration data (contains the calibration factors for the spirometer, as well as paced breathing baseline regression coefficients)

# MERGING DATA
## HORIZONTALLY MERGING EACH PARTICIPANT'S PHYSIOLOGICAL MINUTE EPOCH DATA WITH DETECTED MINUTE POSTURE DATA, VERTICALLY MERGING EACH PARTICIPANT'S DAY1 AND DAY 2 DATA, AND FINALLY VERTICALLY MERGING ALL PARTICIPANT DATA TO GET AN OVERALL DATAFRAME WITH ALL PHYSIOLOGICAL MINUTE EPOCHS AND POSTURES

In [1]:
#import the necessary libraries
import os  #for handling the file directory/path
import pandas as pd
from glob import glob

# specifying the directory/repository that the files are located in 
base_path = r"C:\Users\msa583\OneDrive - Vrije Universiteit Amsterdam\Desktop\RSA Analyses Post-Processing\RQ2"

# identifying the physiological minute epoch xls files that end with _1 or _2 (for the different days)
physio_files = glob(os.path.join(base_path, "*_[12].xls"))

# extract the participant id using the file names (the first part until the first underscore _)
participant_ids = set()
for file in physio_files:
    basename = os.path.basename(file)
    participant_id = basename.split('_')[0]
    participant_ids.add(participant_id)

# define an empty list to collect everything
all_data = []

# looping through each participant (sorted by id)
for pid in sorted(participant_ids, key=int):
    for day in [1, 2]:
        # defining the paths and naming conventions for the physio and posture files
        physio_file = os.path.join(base_path, f"{pid}_{pid}_{day}.xls")
        posture_file = os.path.join(base_path, f"Majoriites per 60sec_{pid}_day{day}.csv")

        # checking if both the physio and the posture files exist
        if os.path.exists(physio_file) and os.path.exists(posture_file):
            try:
                # load the excel physiological data
                physio_df = pd.read_excel(physio_file)

                # loading the csv posture data and choosing only the "Majority_postures" column
                posture_df = pd.read_csv(posture_file, usecols=["Majority_postures"])

                # horizontally -side by side- merging the physio and posture dataframes
                merged_df = pd.concat([physio_df, posture_df], axis=1)

                # adding the participant id and day number as new columns, so the merged_df represents the data for one participant, one day
                merged_df["participant_id"] = pid
                merged_df["day"] = day

                # append the merged dataframe to the all_data list
                all_data.append(merged_df)

            except Exception as e:
                print(f"Error processing participant {pid}, day {day}: {e}")
                
#all day contains the list of all the merged_dfs 
# make all data into one dataframe by vertically stacking the individual dataframes
final_df = pd.concat(all_data, ignore_index=True)

# view the first lines of the overall dataframe
print(final_df.shape)
final_df.head()
print(len(final_df))

(84356, 45)
84356


In [2]:
# save as excel
final_df.to_excel(os.path.join(base_path, "all_participants_merged.xlsx"), index=False)

In [3]:
#calculating the mean absolute percentage error for the spirometer (against the gold standard fixed volume calibration syringe)
#this is a standalone operation that does not affect our main dataframe
import os
import pandas as pd
import numpy as np

# setting the directory including the lab calibration files
calibration_dir = r"C:\Users\msa583\OneDrive - Vrije Universiteit Amsterdam\Desktop\RSA Analyses Post-Processing\RQ2"
calibration_files = [f for f in os.listdir(calibration_dir) if f.startswith("lab_calibration_") and f.endswith(".xlsx")]

# dictionary to store participant percentages
spirometer_percentages = {}

# loop through each file
for file in calibration_files:
    participant_id = os.path.splitext(file)[0].split("_")[-1]
    file_path = os.path.join(calibration_dir, file)
    df = pd.read_excel(file_path)
    
    if "spirometer_estimation_3l" in df.columns:
        percentage = df["spirometer_estimation_3l"].iloc[0]
        spirometer_percentages[participant_id] = percentage
        print(f"Participant {participant_id}: {percentage:.2f}%")
    else:
        print(f"Participant {participant_id}: 'spirometer_estimation_3l' column not found")

# calculate MAPE across the 42 procedures
if spirometer_percentages:
    percentages = np.array(list(spirometer_percentages.values()))
    absolute_errors = np.abs(100 - percentages)
    mape = np.mean(absolute_errors)
    print(f"\nMean Absolute Percentage Error (MAPE): {mape:.2f}%")


Participant 12008: 101.06%
Participant 13304: 99.91%
Participant 15337: 100.89%
Participant 24340: 102.95%
Participant 25201: 98.96%
Participant 25879: 100.09%
Participant 28327: 100.73%
Participant 28404: 101.45%
Participant 30739: 99.96%
Participant 33280: 99.13%
Participant 34714: 98.21%
Participant 34763: 94.68%
Participant 37092: 93.69%
Participant 37818: 100.71%
Participant 38029: 98.72%
Participant 38645: 102.55%
Participant 44066: 96.30%
Participant 46773: 102.41%
Participant 50129: 97.71%
Participant 54783: 100.83%
Participant 55110: 98.03%
Participant 56143: 97.38%
Participant 57458: 99.39%
Participant 58996: 98.29%
Participant 61972: 98.71%
Participant 65919: 101.63%
Participant 66303: 103.39%
Participant 68416: 94.93%
Participant 72950: 96.04%
Participant 74998: 101.06%
Participant 75967: 97.04%
Participant 78692: 97.65%
Participant 81981: 102.83%
Participant 82556: 98.20%
Participant 84645: 95.29%
Participant 88137: 99.49%
Participant 91169: 102.68%
Participant 91850: 102.

# MAKING A CALIBRATED TIDAL VOLUME COLUMN USING THE POSTURE-SPECIFIC CALIBRATION COEFFICIENTS PER PARTICIPANT AND THE PRESENT MINUTE'S POSTURE

In [4]:
import pandas as pd

# file path
file_path = r"C:\Users\msa583\OneDrive - Vrije Universiteit Amsterdam\Desktop\RSA Analyses Post-Processing\RQ2\all_participants_merged.xlsx"

# reading in the excel with the merged participant data
df = pd.read_excel(file_path)

print(len(df))

84356


In [5]:
df.head()

,Location,Physical_Exertion,Posture,Social_Situation,Type_Of_Activity,Location_Code,Physical_Exertion_Code,Posture_Code,Social_Situation_Code,Type_Of_Activity_Code,...,Max_IBI_msec,RSA_msec,Artefact_Free_ECG_percent,Artefact_Free_Respiration_percent,Majority_postures,participant_id,day,Experimental_Conditions,Experimental_Conditions_Code,Artefact_Free_ECG_s
0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,677.0,19.000000,100.0,100.000000,Standing,12008,1,NaN,NaN,NaN
1,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,655.0,8.888889,100.0,100.000000,Standing,12008,1,NaN,NaN,NaN
2,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,692.0,15.000000,100.0,92.487479,Standing,12008,1,NaN,NaN,NaN
3,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,692.0,10.666667,100.0,97.063622,Standing,12008,1,NaN,NaN,NaN
4,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,655.0,15.750000,100.0,92.346090,Standing,12008,1,NaN,NaN,NaN


In [6]:
#calibrating the tidal volumes specific for each posture
import os
import pandas as pd
import numpy as np

# loading all calibration files
calibration_dir = r"C:\Users\msa583\OneDrive - Vrije Universiteit Amsterdam\Desktop\RSA Analyses Post-Processing\RQ2"
calibration_files = [f for f in os.listdir(calibration_dir) if f.startswith("lab_calibration_") and f.endswith(".xlsx")]

# defining an empty dictionary to store the calibration coefficients for each participant
calibration_dict = {}

for file in calibration_files:
    participant_id = os.path.splitext(file)[0].split("_")[-1]
    calib_path = os.path.join(calibration_dir, file)
    calib_df = pd.read_excel(calib_path)
    calibration_dict[participant_id] = calib_df.iloc[0].to_dict()

# function to calculate calibrated tidal volume
def calibrate_tidal_volume(row):
    pid = str(row["participant_id"])
    posture = row["Majority_postures"]
    uncalibrated = row["Tidal_Volume_mOhm"]

    # if there is -9999 in the uncalibrated tidal volume column, put by vu-dams, do not calculate a tidal volume for it
    if uncalibrated == -9999 or pid not in calibration_dict:
        return np.nan

    calib = calibration_dict[pid]

    if posture.lower() == "sitting":
        return uncalibrated * calib["slope_sitting"] + calib["intercept_sitting"]
    elif posture.lower() == "standing":
        return uncalibrated * calib["slope_standing"] + calib["intercept_standing"]
    elif posture.lower() == "lying":
        return uncalibrated * calib["slope_supine"] + calib["intercept_supine"]
    else:
        return np.nan 

# applying the calibration function to the dataframe df
df["Calibrated_Tidal_Volume"] = df.apply(calibrate_tidal_volume, axis=1)

# saving the dataframe
#df.to_excel(os.path.join(calibration_dir, "all_participants_calibrated.xlsx"), index=False)


In [7]:
#datetime indexing the dataframe df, making sure to note the original format is DD-MM-YYYY, with HH:MM:SS:.FFFF
df["datetime"] = pd.to_datetime(
    df["Start_Date"].astype(str) + " " + df["Start_Time"].astype(str),
    format="%d-%m-%Y %H:%M:%S.%f"
)

df.head()

,Location,Physical_Exertion,Posture,Social_Situation,Type_Of_Activity,Location_Code,Physical_Exertion_Code,Posture_Code,Social_Situation_Code,Type_Of_Activity_Code,...,Artefact_Free_ECG_percent,Artefact_Free_Respiration_percent,Majority_postures,participant_id,day,Experimental_Conditions,Experimental_Conditions_Code,Artefact_Free_ECG_s,Calibrated_Tidal_Volume,datetime
0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Standing,12008,1,NaN,NaN,NaN,2.910774,2024-09-02 09:29:53.500
1,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Standing,12008,1,NaN,NaN,NaN,3.063689,2024-09-02 09:30:53.500
2,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,92.487479,Standing,12008,1,NaN,NaN,NaN,2.862958,2024-09-02 09:31:53.500
3,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,97.063622,Standing,12008,1,NaN,NaN,NaN,3.074853,2024-09-02 09:32:53.500
4,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,92.346090,Standing,12008,1,NaN,NaN,NaN,2.223003,2024-09-02 09:33:53.500


In [ ]:
#df.to_excel(os.path.join(calibration_dir, "all_participants_calibrated.xlsx"), index=False)
#saved just to manually double check for some participants that the calibration values are as expected
#verified that this calculation works for all postures, for different participants

# CALCULATING THE MEAN OF MAPE FOR CALIBRATION

In [ ]:
#DO NOT RUN THIS AGAIN (JUST RUNNING ONCE WAS ENOUGH FOR THE STATS)
#finding the mean for the mean absolute percentage error for the calibrations across participants
import os
import pandas as pd

# Set directory path
calibration_dir = r"C:\Users\msa583\OneDrive - Vrije Universiteit Amsterdam\Desktop\RSA Analyses Post-Processing\RQ2"

# List of calibration files
calibration_files = [
    f for f in os.listdir(calibration_dir)
    if f.startswith("lab_calibration_") and f.endswith(".xlsx")
]

# Columns to summarize
columns_to_extract = [
    "mape_supine",
    "mape_sitting",
    "mape_standing",
    "spirometer_estimation_3l"
]

# Collect all calibration rows
calib_rows = []

for file in calibration_files:
    file_path = os.path.join(calibration_dir, file)
    try:
        df = pd.read_excel(file_path)
        calib_rows.append(df[columns_to_extract])
    except Exception as e:
        print(f"Could not read {file}: {e}")

# Concatenate all rows into one DataFrame
all_calib_data = pd.concat(calib_rows, ignore_index=True)

# Calculate summary statistics including count
summary_stats = all_calib_data.agg(["count", "mean", "std", "min", "max"])

# Display results
print(summary_stats)

# REMOVING EPISODES OF LAB RECORDING AND SLEEP SO WE ARE LEFT WITH AWAKE AMBULATORY DATA

In [8]:
# excluding periods of lab time and sleep time
#double-checked all dates and times
exclude_periods = {
    "25879": { #done
        "sleep": [
            ("12-04-2024 22:45:00.000", "13-04-2024 07:40:00.000")
        ],
        "lab_end": "12-04-2024 12:15:00.000"
    },
    "91169": { #done
        "sleep": [
            ("01-05-2024 01:20:00.000", "01-05-2024 09:15:00.000") #the day1 start time in VU-DAMS is an hour later than it is (need to correct for EMA)
        ],
        "lab_end": "30-04-2024 12:55:00.000" #as the VU-AMS recording for day1 for the participant started an hour later than it is, adjusted this
    }, 
    "72950": { #done
        "sleep": [
            ("02-05-2024 01:58:00.000", "02-05-2024 08:43:00.000") 
        ],
        "lab_end": "01-05-2024 12:00:00.000" 
    },
    "65919": {   #done
        "sleep": [
            ("15-05-2024 00:43:00.000", "15-05-2024 08:27:00.000") 
        ],
        "lab_end": "14-05-2024 11:55:00.000" 
    },   
    "33280": {   #done
        "sleep": [
            ("16-05-2024 00:42:00.000", "16-05-2024 08:51:00.000") 
        ],
        "lab_end": "15-05-2024 12:10:00.000" 
    }, 
    "75967": {   #done
        "sleep": [
            ("17-05-2024 00:14:00.000", "17-05-2024 07:49:00.000") 
        ],
        "lab_end": "16-05-2024 11:55:00.000" 
    },
    "38645": {   #done
        "sleep": [
            ("24-05-2024 04:50:00.000", "24-05-2024 12:47:00.000") 
        ],
        "lab_end": "23-05-2024 12:16:00.000" 
    },
    "92674": {   #done
        "sleep": [
            ("30-05-2024 00:55:55.000", "30-05-2024 08:54:00.000") 
        ],
        "lab_end": "29-05-2024 12:04:00.000" 
    },
    "37092": {   #done
        "sleep": [
            ("31-05-2024 00:04:00.000", "01-06-2024 07:52:00.000")  #the day2 should actually be 31-5-2024 and not the first of june, correct it for EMA
        ],  #the dates here are correct, the VU-DAMS jumped from 31st of May around 2:40 am to 1st of June 2:40 am
        "lab_end": "30-05-2024 12:22:00.000" 
    },    
    "57458": {   #done
        "sleep": [
            ("11-06-2024 22:35:00.000", "12-06-2024 08:22:00.000")  
        ],
        "lab_end": "11-06-2024 11:54:00.000" 
    },  
    "91850": {   #done
        "sleep": [
            ("14-06-2024 01:07:00.000", "14-06-2024 09:54:00.000")  
        ],
        "lab_end": "13-06-2024 11:47:00.000" 
    },
    "81981": {   #done
        "sleep": [
            ("20-06-2024 01:14:00.000", "20-06-2024 08:52:00.000")  
        ],
        "lab_end": "19-06-2024 12:03:00.000" 
    },    
    "61972": {   #done
        "sleep": [
            ("25-06-2024 01:47:00.000", "25-06-2024 08:33:00.000")  
        ],
        "lab_end": "24-06-2024 11:56:00.000" 
    },    
    "15337": {   #this participant does not have the sleep time data, so that part is not applicable - done
        
        "lab_end": "25-06-2024 12:13:00.000" 
    },
    "50129": {    #done
        "sleep": [
            ("26-06-2024 23:57:00.000", "27-06-2024 06:48:00.000")  
        ],
        "lab_end": "26-06-2024 12:31:00.000" 
    },
    "92957": {    #done
        "sleep": [
            ("04-07-2024 00:38:00.000", "04-07-2024 08:28:00.000")  
        ],
        "lab_end": "03-07-2024 12:17:00.000" 
    },   
    "28327": {    #done
        "sleep": [
            ("05-07-2024 02:01:00.000", "05-07-2024 09:29:00.000")  
        ],
        "lab_end": "04-07-2024 12:15:00.000" 
    },
    "46773": {    #done
        "sleep": [
            ("09-07-2024 22:55:00.000", "10-07-2024 06:28:00.000")  
        ],
        "lab_end": "09-07-2024 11:00:00.000" #the day1 recording start time is written as an hour earlier than it actually is, so also adjusted this
    },
    "30739": {  #done  
        "sleep": [
            ("11-07-2024 23:36:00.000", "12-07-2024 08:15:00.000")  
        ],
        "lab_end": "11-07-2024 10:57:00.000" #the day1 recording start time is written as an hour earlier than it actually is, so also adjusted this
    },    
    "93676": {    #done
        "sleep": [
            ("06-08-2024 21:15:00.000", "07-08-2024 13:17:00.000")  
        ],
        "lab_end": "06-08-2024 11:29:00.000" #the day1 recording start time is written as an hour earlier than it actually is, so also adjusted this
    },
    "13304": {   #this participant does not have the sleep time data, so that part is not applicable - done
        
        "lab_end": "09-08-2024 11:59:00.000" 
    },    
    "37818": {    #done
        "sleep": [
            ("20-08-2024 21:22:00.000", "21-08-2024 05:18:00.000")  
        ],
        "lab_end": "20-08-2024 10:50:00.000"  #the day1 recording start time is written as an hour earlier than it actually is, so also adjusted this
    },    
    "12008": {    #done
        "sleep": [
            ("02-09-2024 22:02:00.000", "03-09-2024 07:41:00.000")  
        ],
        "lab_end": "02-09-2024 11:09:00.000"   #the day1 recording start time is written as an hour earlier than it actually is, so also adjusted this
    },
    "25201": {    #done
        "sleep": [
            ("10-09-2024 23:14:00.000", "11-09-2024 08:23:00.000")  
        ],
        "lab_end": "10-09-2024 11:17:00.000"  #the day1 recording start time is written as an hour earlier than it actually is, so also adjusted this
    },   
    "38029": {    #done
        "sleep": [
            ("11-09-2024 23:32:00.000", "12-09-2024 07:34:00.000")  
        ],
        "lab_end": "11-09-2024 12:05:00.000" 
    },
    "24340": {    #done
        "sleep": [
            ("18-09-2024 01:23:00.000", "18-09-2024 07:48:00.000")  
        ],
        "lab_end": "17-09-2024 12:28:00.000" 
    },
    "58996": {    #done
        "sleep": [
            ("20-09-2024 01:13:00.000", "20-09-2024 08:14:00.000")  
        ],
        "lab_end": "19-09-2024 12:11:00.000" 
    },
    "88137": {    #done
        "sleep": [
            ("21-09-2024 00:00:49.000", "21-09-2024 09:58:00.000")  
        ],
        "lab_end": "20-09-2024 10:50:00.000"  #the day1 recording start time is written as an hour earlier than it actually is, so also adjusted this
    },
    "56143": {   #done
        "sleep": [
            ("27-09-2024 01:04:00.000", "27-09-2024 07:45:00.000")  
        ],
        "lab_end": "26-09-2024 11:50:00.000" 
    },
    "28404": {  #done
        "sleep": [
            ("03-10-2024 23:59:00.000", "04-10-2024 07:26:00.000")  
        ],
        "lab_end": "03-10-2024 12:30:00.000" 
    },    
    "68416": {  #done
        "sleep": [
            ("01-11-2024 02:13:00.000", "01-11-2024 10:38:00.000")  
        ],
        "lab_end": "31-10-2024 11:56:00.000" 
    }, 
    "44066": {   #done
        "sleep": [
            ("02-11-2024 00:10:20.000", "02-11-2024 09:02:00.000")  
        ],
        "lab_end": "01-11-2024 12:10:00.000" 
    },   
    "34714": {   #done
        "sleep": [
            ("08-11-2024 01:16:00.000", "08-11-2024 09:30:00.000")  
        ],
        "lab_end": "07-11-2024 12:23:00.000" 
    },   
    "54783": {   #done
        "sleep": [
            ("13-11-2024 23:05:00.000", "14-11-2024 00:42:00.000")  #as the end of sleep put in some time after the end (00:41:54.658) of this file
        ],
        "lab_end": "13-11-2024 12:10:00.000" 
    },     
    "74998": {   #done
        "sleep": [
            ("15-11-2024 03:27:00.000", "15-11-2024 08:24:00.000")  
        ],
        "lab_end": "14-11-2024 12:17:00.000" 
    },
     "82556": {   #done
        "sleep": [
            ("15-11-2024 22:52:00.000", "16-11-2024 07:28:53.000")  
        ],
        "lab_end": "15-11-2024 12:24:00.000" 
    },   
     "78692": {   #done
        "sleep": [
            ("06-12-2024 00:16:00.000", "06-12-2024 10:32:00.000")  
        ],
        "lab_end": "05-12-2024 12:07:00.000" 
    },  
     "55110": {   #done
        "sleep": [
            ("16-01-2025 23:11:00.000", "17-01-2025 06:29:00.000")  
        ],
        "lab_end": "16-01-2025 12:14:00.000" 
    }, 
     "84645": {  #done
        "sleep": [
            ("25-01-2025 00:09:00.000", "25-01-2025 09:31:00.000")  
        ],
        "lab_end": "24-01-2025 11:50:00.000" 
    },
     "34763": {  #done
        "sleep": [
            ("29-01-2025 03:27:00.000", "29-01-2025 09:10:00.000")  
        ],
        "lab_end": "28-01-2025 12:16:00.000" 
    },
     "66303": {  #done
        "sleep": [
            ("04-02-2025 01:04:00.000", "04-02-2025 08:16:00.000")  
        ],
        "lab_end": "03-02-2025 12:14:00.000" 
    },
     "99857": {  #done
        "sleep": [
            ("12-02-2025 00:24:00.000", "12-02-2025 08:42:00.000")  
        ],
        "lab_end": "11-02-2025 11:50:00.000" 
    }
}


In [9]:
filtered_dfs = []  # list to collect the awake ambulatory data for all participants

# obtain the list of unique participant ids
all_participants = df["participant_id"].astype(str).unique()

for pid in all_participants:
    # selecting and copying the given participant's data
    participant_data = df[df["participant_id"].astype(str) == pid].copy()

    # if the participant is in the exclude list, apply the filters
    if pid in exclude_periods:
        periods = exclude_periods[pid]

        # if lab_end is provided, remove the lab data
        if "lab_end" in periods:
            lab_end = pd.to_datetime(periods["lab_end"], format="%d-%m-%Y %H:%M:%S.%f")
            participant_data = participant_data[participant_data["datetime"] > lab_end]  #keeps data with datetime later than end of lab procedure

        # if sleep period is provided, remove the sleep data
        if "sleep" in periods:
            for start_str, end_str in periods["sleep"]:
                sleep_start = pd.to_datetime(start_str, format="%d-%m-%Y %H:%M:%S.%f")
                sleep_end = pd.to_datetime(end_str, format="%d-%m-%Y %H:%M:%S.%f")
                participant_data = participant_data[
                    ~((participant_data["datetime"] >= sleep_start) & (participant_data["datetime"] <= sleep_end))   #removing interval of sleep
                ]

    # add the filtered data to the full list
    filtered_dfs.append(participant_data)

# combine the contents of the filtered_dfs list into one general dataframe
df_awake_ambulatory = pd.concat(filtered_dfs, ignore_index=True)


In [10]:
df_awake_ambulatory  #this is the dataset that does not have the lab recording and the sleep episodes anymore

,Location,Physical_Exertion,Posture,Social_Situation,Type_Of_Activity,Location_Code,Physical_Exertion_Code,Posture_Code,Social_Situation_Code,Type_Of_Activity_Code,...,Artefact_Free_ECG_percent,Artefact_Free_Respiration_percent,Majority_postures,participant_id,day,Experimental_Conditions,Experimental_Conditions_Code,Artefact_Free_ECG_s,Calibrated_Tidal_Volume,datetime
0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Standing,12008,1,NaN,NaN,NaN,2.933965,2024-09-02 11:09:53.500
1,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Standing,12008,1,NaN,NaN,NaN,3.128341,2024-09-02 11:10:53.500
2,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,93.531469,Standing,12008,1,NaN,NaN,NaN,3.182655,2024-09-02 11:11:53.500
3,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,92.500000,Standing,12008,1,NaN,NaN,NaN,3.171272,2024-09-02 11:12:53.500
4,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Sitting,12008,1,NaN,NaN,NaN,1.706549,2024-09-02 11:13:53.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61485,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.000000,Sitting,99857,2,-9999.0,-9999.0,NaN,1.830138,2025-02-12 21:55:39.500
61486,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.000000,Sitting,99857,2,-9999.0,-9999.0,NaN,1.581758,2025-02-12 21:56:39.500
61487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.000000,Sitting,99857,2,-9999.0,-9999.0,NaN,1.416549,2025-02-12 21:57:39.500
61488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,81.878089,Sitting,99857,2,-9999.0,-9999.0,NaN,2.225654,2025-02-12 21:58:39.500


In [11]:
import numpy as np

# filtering out rows where Total_Motility_mg >= 160 - there is at least a physical intensity akin to that of walking
initial_row_count = len(df_awake_ambulatory)

df_awake_ambulatory = df_awake_ambulatory[df_awake_ambulatory['Total_Motility_mg'] < 160] #creating the no physical activity version of the df

removed_rows = initial_row_count - len(df_awake_ambulatory)
percentage_removed = (removed_rows / initial_row_count) * 100
print(f"Percentage of rows removed (Total_Motility_mg >= 160): {percentage_removed:.2f}%")
df_awake_ambulatory #this is the dataframe with no ambulatory, no lab, no medium-intense physical activity data

Percentage of rows removed (Total_Motility_mg >= 160): 9.37%


,Location,Physical_Exertion,Posture,Social_Situation,Type_Of_Activity,Location_Code,Physical_Exertion_Code,Posture_Code,Social_Situation_Code,Type_Of_Activity_Code,...,Artefact_Free_ECG_percent,Artefact_Free_Respiration_percent,Majority_postures,participant_id,day,Experimental_Conditions,Experimental_Conditions_Code,Artefact_Free_ECG_s,Calibrated_Tidal_Volume,datetime
0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Standing,12008,1,NaN,NaN,NaN,2.933965,2024-09-02 11:09:53.500
4,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Sitting,12008,1,NaN,NaN,NaN,1.706549,2024-09-02 11:13:53.500
5,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,94.863563,Standing,12008,1,NaN,NaN,NaN,2.811731,2024-09-02 11:14:53.500
12,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Standing,12008,1,NaN,NaN,NaN,2.683670,2024-09-02 11:21:53.500
13,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,96.875000,Standing,12008,1,NaN,NaN,NaN,2.478718,2024-09-02 11:22:53.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61485,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.000000,Sitting,99857,2,-9999.0,-9999.0,NaN,1.830138,2025-02-12 21:55:39.500
61486,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.000000,Sitting,99857,2,-9999.0,-9999.0,NaN,1.581758,2025-02-12 21:56:39.500
61487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.000000,Sitting,99857,2,-9999.0,-9999.0,NaN,1.416549,2025-02-12 21:57:39.500
61488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,81.878089,Sitting,99857,2,-9999.0,-9999.0,NaN,2.225654,2025-02-12 21:58:39.500


In [ ]:
#df_awake_ambulatory.to_excel('df_awake_ambulatory_checkexclusions.xlsx')
#saved only to make sure the sleep and lab period and physically active periods are eliminated (manually checked some rows in the excel) - correct

In [ ]:
#let's calculate for this duration of recording the percentage of artifacts for the ECG and respiration 

In [12]:
# check if there are any 0s at all in Artefact_Free_ECG_percent
zero_values = df_awake_ambulatory[
   
    (df_awake_ambulatory['Artefact_Free_ECG_percent'] == 0)
]

print(zero_values)


Empty DataFrame
Columns: [Location, Physical_Exertion, Posture, Social_Situation, Type_Of_Activity, Location_Code, Physical_Exertion_Code, Posture_Code, Social_Situation_Code, Type_Of_Activity_Code, Subject_ID, Label_ID, Start_Date, Start_Time, End_Date, End_Time, Label_Duration_s, Total_Motility_mg, Average_IBI_msec, Average_HR_bpm, SDNN_msec, Respiration_Rate_bpm, Tidal_Volume_mOhm, RMSSD_msec, LF_ms, RSA0_msec, HF_ms, PEP_msec, LVET_msec, TWave_amplitude_mV, Stroke_Volume_(Nederend_2017)_cc, Minute_Volume_(Nederend_2017)_lmin, Average_SCL_uS, nsSCRs_per_minute_ppm, Min_IBI_msec, Max_IBI_msec, RSA_msec, Artefact_Free_ECG_percent, Artefact_Free_Respiration_percent, Majority_postures, participant_id, day, Experimental_Conditions, Experimental_Conditions_Code, Artefact_Free_ECG_s, Calibrated_Tidal_Volume, datetime]
Index: []

[0 rows x 47 columns]


In [13]:
#In the artifact free ECG column, there are not any 0s. When there is no data for ECG, it is specified with a value of -9999. We will thus replace the
#-9999 values in this column with 0 (meaning there is a lack of artifact free ECG)
df_awake_ambulatory['Artefact_Free_ECG_percent'] = df_awake_ambulatory['Artefact_Free_ECG_percent'].replace(-9999, 0)

# mean of Artefact_Free_ECG_percent
mean_ecg = df_awake_ambulatory['Artefact_Free_ECG_percent'].mean()

print("Mean Artefact Free ECG Percent:", mean_ecg)


Mean Artefact Free ECG Percent: 87.4919473432547


C:\Users\msa583\AppData\Local\Temp\ipykernel_31700\2289913148.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_awake_ambulatory['Artefact_Free_ECG_percent'] = df_awake_ambulatory['Artefact_Free_ECG_percent'].replace(-9999, 0)


In [14]:
#do the same for respiration
df_awake_ambulatory['Artefact_Free_Respiration_percent'] = df_awake_ambulatory['Artefact_Free_Respiration_percent'].replace(-9999, 0)

# mean of Artefact_Free_ECG_percent
mean_resp = df_awake_ambulatory['Artefact_Free_Respiration_percent'].mean()

print("Mean Artefact Free Respiration Percent:", mean_resp)

Mean Artefact Free Respiration Percent: 85.78365557741692


C:\Users\msa583\AppData\Local\Temp\ipykernel_31700\1931996242.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_awake_ambulatory['Artefact_Free_Respiration_percent'] = df_awake_ambulatory['Artefact_Free_Respiration_percent'].replace(-9999, 0)


In [15]:
# calculate the mean, min, and max for the columns of RSA0, RR, IBI, and Calibrated_Tidal_Volume
rsa0_mean = df_awake_ambulatory['RSA0_msec'].mean()
rsa0_min = df_awake_ambulatory['RSA0_msec'].min()
rsa0_max = df_awake_ambulatory['RSA0_msec'].max()

respiration_rate_mean = df_awake_ambulatory['Respiration_Rate_bpm'].mean()
respiration_rate_min = df_awake_ambulatory['Respiration_Rate_bpm'].min()
respiration_rate_max = df_awake_ambulatory['Respiration_Rate_bpm'].max()

ibi_mean = df_awake_ambulatory['Average_IBI_msec'].mean()
ibi_min = df_awake_ambulatory['Average_IBI_msec'].min()
ibi_max = df_awake_ambulatory['Average_IBI_msec'].max()

vt_mean = df_awake_ambulatory['Calibrated_Tidal_Volume'].mean()
vt_min = df_awake_ambulatory['Calibrated_Tidal_Volume'].min()
vt_max = df_awake_ambulatory['Calibrated_Tidal_Volume'].max()

# display the results
print("RSA0_msec - Mean:", rsa0_mean, "Min:", rsa0_min, "Max:", rsa0_max)
print("Respiration_Rate_bpm - Mean:", respiration_rate_mean, "Min:", respiration_rate_min, "Max:", respiration_rate_max)
print("Average_IBI_msec - Mean:", ibi_mean, "Min:", ibi_min, "Max:", ibi_max)
print("Calibrated_Tidal_Volume - Mean:", vt_mean, "Min:", vt_min, "Max:", vt_max)


RSA0_msec - Mean: -1140.4659980705615 Min: -9999.0 Max: 415.8571428571428
Respiration_Rate_bpm - Mean: -1170.7933238997928 Min: -9999.0 Max: 33.02552952430968
Average_IBI_msec - Mean: -465.176894823439 Min: -9999.0 Max: 1334.875
Calibrated_Tidal_Volume - Mean: 0.7337880595216868 Min: -22.774494504921062 Max: 84.60541128778387


In [16]:
# we only select rows in which there is more than 45% artifact-free data for both Artefact_Free_ECG_percent and Artefact_Free_Respiration_percent
#otherwise we would still have -9999 (NA) values for RSA0, IBI, and RR (parameters of interest)
df_awake_ambulatory = df_awake_ambulatory[
    (df_awake_ambulatory['Artefact_Free_ECG_percent'] >= 45) &
    (df_awake_ambulatory['Artefact_Free_Respiration_percent'] >= 45)
]
df_awake_ambulatory

,Location,Physical_Exertion,Posture,Social_Situation,Type_Of_Activity,Location_Code,Physical_Exertion_Code,Posture_Code,Social_Situation_Code,Type_Of_Activity_Code,...,Artefact_Free_ECG_percent,Artefact_Free_Respiration_percent,Majority_postures,participant_id,day,Experimental_Conditions,Experimental_Conditions_Code,Artefact_Free_ECG_s,Calibrated_Tidal_Volume,datetime
0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Standing,12008,1,NaN,NaN,NaN,2.933965,2024-09-02 11:09:53.500
4,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Sitting,12008,1,NaN,NaN,NaN,1.706549,2024-09-02 11:13:53.500
5,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,94.863563,Standing,12008,1,NaN,NaN,NaN,2.811731,2024-09-02 11:14:53.500
12,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,100.000000,Standing,12008,1,NaN,NaN,NaN,2.683670,2024-09-02 11:21:53.500
13,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,...,100.0,96.875000,Standing,12008,1,NaN,NaN,NaN,2.478718,2024-09-02 11:22:53.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61485,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.000000,Sitting,99857,2,-9999.0,-9999.0,NaN,1.830138,2025-02-12 21:55:39.500
61486,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.000000,Sitting,99857,2,-9999.0,-9999.0,NaN,1.581758,2025-02-12 21:56:39.500
61487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,100.000000,Sitting,99857,2,-9999.0,-9999.0,NaN,1.416549,2025-02-12 21:57:39.500
61488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,100.0,81.878089,Sitting,99857,2,-9999.0,-9999.0,NaN,2.225654,2025-02-12 21:58:39.500


In [17]:
# calculate the mean, min, and max for the columns of RSA0, RR, IBI to make sure that following our selection of columns, the -9999s are not there
rsa0_mean = df_awake_ambulatory['RSA0_msec'].mean()
rsa0_min = df_awake_ambulatory['RSA0_msec'].min()
rsa0_max = df_awake_ambulatory['RSA0_msec'].max()

respiration_rate_mean = df_awake_ambulatory['Respiration_Rate_bpm'].mean()
respiration_rate_min = df_awake_ambulatory['Respiration_Rate_bpm'].min()
respiration_rate_max = df_awake_ambulatory['Respiration_Rate_bpm'].max()

ibi_mean = df_awake_ambulatory['Average_IBI_msec'].mean()
ibi_min = df_awake_ambulatory['Average_IBI_msec'].min()
ibi_max = df_awake_ambulatory['Average_IBI_msec'].max()

# display the results
print("RSA0_msec - Mean:", rsa0_mean, "Min:", rsa0_min, "Max:", rsa0_max)
print("Respiration_Rate_bpm - Mean:", respiration_rate_mean, "Min:", respiration_rate_min, "Max:", respiration_rate_max)
print("Average_IBI_msec - Mean:", ibi_mean, "Min:", ibi_min, "Max:", ibi_max)


RSA0_msec - Mean: 57.21108640251692 Min: 0.0 Max: 415.8571428571428
Respiration_Rate_bpm - Mean: 17.520442732572874 Min: 7.279411764705882 Max: 33.02552952430968
Average_IBI_msec - Mean: 735.5681619471582 Min: 340.2732558139535 Max: 1334.875


In [18]:
# calculate the mean, min, and max for the columns of RSA0, RR, IBI, and Calibrated_Tidal_Volume
rsa0_mean = df_awake_ambulatory['RSA0_msec'].mean()
rsa0_min = df_awake_ambulatory['RSA0_msec'].min()
rsa0_max = df_awake_ambulatory['RSA0_msec'].max()

respiration_rate_mean = df_awake_ambulatory['Respiration_Rate_bpm'].mean()
respiration_rate_min = df_awake_ambulatory['Respiration_Rate_bpm'].min()
respiration_rate_max = df_awake_ambulatory['Respiration_Rate_bpm'].max()

ibi_mean = df_awake_ambulatory['Average_IBI_msec'].mean()
ibi_min = df_awake_ambulatory['Average_IBI_msec'].min()
ibi_max = df_awake_ambulatory['Average_IBI_msec'].max()

vt_mean = df_awake_ambulatory['Calibrated_Tidal_Volume'].mean()
vt_min = df_awake_ambulatory['Calibrated_Tidal_Volume'].min()
vt_max = df_awake_ambulatory['Calibrated_Tidal_Volume'].max()

# display the results
print("RSA0_msec - Mean:", rsa0_mean, "Min:", rsa0_min, "Max:", rsa0_max)
print("Respiration_Rate_bpm - Mean:", respiration_rate_mean, "Min:", respiration_rate_min, "Max:", respiration_rate_max)
print("Average_IBI_msec - Mean:", ibi_mean, "Min:", ibi_min, "Max:", ibi_max)
print("Calibrated_Tidal_Volume - Mean:", vt_mean, "Min:", vt_min, "Max:", vt_max)


RSA0_msec - Mean: 57.21108640251692 Min: 0.0 Max: 415.8571428571428
Respiration_Rate_bpm - Mean: 17.520442732572874 Min: 7.279411764705882 Max: 33.02552952430968
Average_IBI_msec - Mean: 735.5681619471582 Min: 340.2732558139535 Max: 1334.875
Calibrated_Tidal_Volume - Mean: 0.716032707767372 Min: -22.774494504921062 Max: 71.08422386075979


In [19]:
# calculate mean, min, and max for Calibrated_Tidal_Volume, after the removal of outliers for each participant specifically
participant_stats = df_awake_ambulatory.groupby('participant_id')['Calibrated_Tidal_Volume'].agg(['mean', 'min', 'max'])
print(participant_stats)

                    mean        min        max
participant_id                                
12008           1.700330 -12.218596   3.295508
13304           0.617628   0.096279   9.002533
15337           0.681023   0.371967   3.637552
24340           0.423655  -1.092138  14.241041
25201           0.481401   0.118884   1.426210
25879           0.520385   0.207447   3.794518
28327           0.745354  -0.416675   5.765957
28404           0.631897  -0.002496  10.017951
30739           0.565304   0.048086   5.143306
33280           0.365867  -0.025222   3.298081
34714           0.780869   0.466584   3.625991
34763           0.676459   0.230354  16.754334
37092           0.878177   0.499509  13.488824
37818           0.842010   0.485022   4.998923
38029           0.571333   0.336559   1.290300
38645           0.539844   0.378149   2.616565
44066           0.413442   0.109487  13.181655
46773           0.348449  -3.044703   0.612588
50129           0.409418   0.299495   0.781693
54783        

In [20]:
#filtering out of epochs with implausible and deviant tidal volume values

# filtering out rows with implausible calibrated volume values - specifically, a Calibrated_Tidal_Volume of less than 0.1 or greater than 6.5 L
initial_row_count = len(df_awake_ambulatory)

df_awake_ambulatory = df_awake_ambulatory[
    (df_awake_ambulatory['Calibrated_Tidal_Volume'] >= 0.1) & 
    (df_awake_ambulatory['Calibrated_Tidal_Volume'] <= 6.5)
]

removed_rows = initial_row_count - len(df_awake_ambulatory)

percentage_removed = (removed_rows / initial_row_count) * 100

print(f"Percentage of rows removed (Calibrated_Tidal_Volume < 0 or > 6.5): {percentage_removed:.2f}%")

# grouping per participant_id and per Majority_postures, then calculate mean and sd for Calibrated_Tidal_Volume
stats = df_awake_ambulatory.groupby(['participant_id', 'Majority_postures'])['Calibrated_Tidal_Volume'].agg(['mean', 'std'])

# function to remove outliers for each row based on its participant_id AND posture
def remove_outliers(row, stats):
    # calculate the mean and sd for the given participant_id and Majority_postures
    mean = stats.loc[(row['participant_id'], row['Majority_postures']), 'mean']
    std = stats.loc[(row['participant_id'], row['Majority_postures']), 'std']
    
    # calculate distance from mean per each tidal volume value
    distance_from_mean = abs(row['Calibrated_Tidal_Volume'] - mean) / std
    # if the distance is more than 3 standard deviations, enter NaN
    if distance_from_mean >= 3:
        return np.nan  
    return row['Calibrated_Tidal_Volume']

# run the remove_outliers function row by row
initial_row_count = len(df_awake_ambulatory)

df_awake_ambulatory['Cleaned_Tidal_Volume'] = df_awake_ambulatory.apply(lambda row: remove_outliers(row, stats), axis=1)

# actually drop rows where Cleaned_Tidal_Volume is NaN 
removed_rows = df_awake_ambulatory['Cleaned_Tidal_Volume'].isna().sum()
percentage_removed = (removed_rows / initial_row_count) * 100
df_awake_ambulatory = df_awake_ambulatory.dropna(subset=['Cleaned_Tidal_Volume'])

print(f"Percentage of rows removed (outliers more than 3 SDs away): {percentage_removed:.2f}%")

print(f"Remaining rows: {len(df_awake_ambulatory)}")
print(df_awake_ambulatory)

Percentage of rows removed (Calibrated_Tidal_Volume < 0 or > 6.5): 1.26%
Percentage of rows removed (outliers more than 3 SDs away): 1.34%
Remaining rows: 47338
       Location  Physical_Exertion  Posture  Social_Situation  \
0       -9999.0            -9999.0  -9999.0           -9999.0   
4       -9999.0            -9999.0  -9999.0           -9999.0   
5       -9999.0            -9999.0  -9999.0           -9999.0   
12      -9999.0            -9999.0  -9999.0           -9999.0   
13      -9999.0            -9999.0  -9999.0           -9999.0   
...         ...                ...      ...               ...   
61485       NaN                NaN      NaN               NaN   
61486       NaN                NaN      NaN               NaN   
61487       NaN                NaN      NaN               NaN   
61488       NaN                NaN      NaN               NaN   
61489       NaN                NaN      NaN               NaN   

       Type_Of_Activity  Location_Code  Physical_Exertion_

C:\Users\msa583\AppData\Local\Temp\ipykernel_31700\1678507945.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_awake_ambulatory['Cleaned_Tidal_Volume'] = df_awake_ambulatory.apply(lambda row: remove_outliers(row, stats), axis=1)


In [21]:
# calculate mean, min, and max for Calibrated_Tidal_Volume, after the removal of outliers for each participant specifically
participant_stats = df_awake_ambulatory.groupby('participant_id')['Calibrated_Tidal_Volume'].agg(['mean', 'min', 'max'])
print(participant_stats)

                    mean       min       max
participant_id                              
12008           1.712384  0.457229  3.295508
13304           0.568085  0.118275  1.303695
15337           0.665486  0.371967  1.221897
24340           0.460630  0.143540  1.056040
25201           0.472315  0.118884  0.917020
25879           0.514647  0.207447  1.167206
28327           0.820621  0.101613  3.396424
28404           0.614978  0.102459  1.338234
30739           0.561642  0.100385  1.787948
33280           0.365262  0.101986  1.207562
34714           0.775036  0.466584  1.396584
34763           0.508627  0.230354  2.466209
37092           0.771588  0.499509  1.715267
37818           0.829614  0.485022  1.195109
38029           0.567331  0.336559  1.020966
38645           0.534450  0.378149  0.913602
44066           0.390183  0.109487  0.922843
46773           0.354813  0.193881  0.541578
50129           0.407822  0.299495  0.717867
54783           0.869729  0.780386  1.144406
55110     

In [22]:
# calculate mean, min, and max per posture for IBI, RSA0, vector magnitude, tidal volume, and respiration rate
summary_stats = df_awake_ambulatory.groupby('Majority_postures')[['Average_IBI_msec', 'RSA0_msec', 'Total_Motility_mg', 'Calibrated_Tidal_Volume', 'Respiration_Rate_bpm']].agg(['mean', 'min', 'max'])

# calculate the percentage of rows per posture
posture_counts = df_awake_ambulatory['Majority_postures'].value_counts()
total_rows = len(df_awake_ambulatory)
posture_percentage = (posture_counts / total_rows) * 100

print("Summary Statistics (Mean, Min, Max) by Posture:")
print(summary_stats)

print("\nPercentage of Rows for Each Posture:")
print(posture_percentage)



Summary Statistics (Mean, Min, Max) by Posture:
                  Average_IBI_msec                           RSA0_msec       \
                              mean         min          max       mean  min   
Majority_postures                                                             
Lying                   799.804252  395.708609  1334.875000  62.300921  0.0   
Sitting                 757.520838  346.005814  1306.511111  62.108393  0.0   
Standing                649.156875  340.273256  1122.038462  42.212770  0.0   

                              Total_Motility_mg                        \
                          max              mean       min         max   
Majority_postures                                                       
Lying              305.555556         24.806465  3.456337  159.753184   
Sitting            415.857143         31.355670  3.405684  159.978562   
Standing           322.444444         74.692697  5.411640  159.968723   

                  Calibrated_Tidal_Vol

In [ ]:
df_awake_ambulatory #there are 47338 rows of usable data that equates to 789 hours of data in total
#the predominant posture is sitting, as expected, followed by standing and lying

In [23]:
#need to save the excel at this point as the Notebook freezes otherwise (100% CPU)
df_awake_ambulatory.to_excel("df_awake_ambulatory_noRSAmetricsyet.xlsx")

# MAKING THE MULTIPLE REGRESSIONS IN WHICH RSA0 IS PREDICTED BY RESPIRATION RATE AND CALIBRATED TIDAL VOLUME PER POSTURE PER PARTICIPANT

In [24]:
#clean all outputs and reupload the dataframe (to save memory) - also do not print the dataframe as this consumes memory
#if you need to check something, save it as an excel and check on excel
df_awake_ambulatory = pd.read_excel("df_awake_ambulatory_noRSAmetricsyet.xlsx")

In [25]:
#print the number of data points per participant per posture
import pandas as pd

# showing full output, displaying row truncation
pd.set_option('display.max_rows', None) 
pd.set_option('display.max_columns', None)  

# group by participant_id and posture to count the number of epochs
epoch_counts = df_awake_ambulatory.groupby(['participant_id', 'Majority_postures']).size()
print(epoch_counts)


participant_id  Majority_postures
12008           Lying                 194
                Sitting               554
                Standing              374
13304           Lying                  16
                Sitting               122
                Standing               51
15337           Lying                  26
                Sitting               335
                Standing              168
24340           Lying                   1
                Sitting              1045
                Standing              259
25201           Lying                  78
                Sitting               738
                Standing              346
25879           Lying                  57
                Sitting               954
                Standing              237
28327           Lying                  86
                Sitting              1019
                Standing               93
28404           Lying                  36
                Sitting              1035


In [ ]:
# make a dataframe with the Standing data of participant 12008 to double check if the regression is correct when done with the function
#checked, the function that makes the participant and posture specific multiple regression lines for Approach 3 (for getting at residual RSA) is correct
#df_standing_12008 = df_awake_ambulatory[
  #  (df_awake_ambulatory['participant_id'] == 12008) &
  #  (df_awake_ambulatory['Majority_postures'] == 'Standing')
#]

#print(df_standing_12008)
#print(len(df_standing_12008))

In [ ]:
#import statsmodels.api as sm

# Step 1: Define the predictors (X) and outcome (y)
#X = df_standing_12008[['Calibrated_Tidal_Volume', 'Respiration_Rate_bpm']]
#y = df_standing_12008['RSA0_msec']

# Step 2: Add a constant (intercept) to the predictors
#X = sm.add_constant(X)

# Step 3: Fit the regression model
#model = sm.OLS(y, X).fit()

# Step 4: Print the regression summary
#print(model.summary())


In [26]:
import statsmodels.api as sm
import numpy as np

# function to run the multiple regressions for Approach 3 and subsequently calculate the Residual RSAs
# when there are less than 20 data points for a participant's given posture category, it skips that line (not statistically apppropriate to regress then)
def run_regression_and_calculate_residual(group, df_awake_ambulatory):
    # check if a min of 20 points is available
    if len(group) < 20:
        print(f"Skipping regression for Participant {group['participant_id'].iloc[0]} - Posture: {group['Majority_postures'].iloc[0]} (Less than 20 data points)\n")
        # put NaN to RSA0_residual since no regression is available
        df_awake_ambulatory.loc[group.index, 'RSA0_residual'] = np.nan
        return df_awake_ambulatory  
    
    # defining the independent variables
    X = group[['Calibrated_Tidal_Volume', 'Respiration_Rate_bpm']]
    # defining the dependent variable
    y = group['RSA0_msec']
    
    # add a constant for the intercept
    X = sm.add_constant(X)
    
    # fit the regression model
    model = sm.OLS(y, X).fit()
    
    # find the predicted RSA0 values
    group['predicted_RSA0'] = model.predict(X)
    
    # calculating the residual: observed - predicted
    group['RSA0_residual'] = group['RSA0_msec'] - group['predicted_RSA0']
    
    # adding the residual to the dataframe
    df_awake_ambulatory.loc[group.index, 'RSA0_residual'] = group['RSA0_residual']
    
    # print the equation for the regression
    intercept = model.params['const']
    tidal_volume_coef = model.params['Calibrated_Tidal_Volume']
    respiration_rate_coef = model.params['Respiration_Rate_bpm']
    
    intercept_pvalue = model.pvalues['const']
    tidal_volume_pvalue = model.pvalues['Calibrated_Tidal_Volume']
    respiration_rate_pvalue = model.pvalues['Respiration_Rate_bpm']
    
    print(f"Regression Equation for Participant {group['participant_id'].iloc[0]} - Posture: {group['Majority_postures'].iloc[0]}")
    print(f"RSA0_msec = {intercept:.4f} + ({tidal_volume_coef:.4f} * Calibrated_Tidal_Volume) + ({respiration_rate_coef:.4f} * Respiration_Rate_bpm)")
    print(f"P-value for Intercept: {intercept_pvalue:.4f}")
    print(f"P-value for Calibrated_Tidal_Volume: {tidal_volume_pvalue:.4f}")
    print(f"P-value for Respiration_Rate_bpm: {respiration_rate_pvalue:.4f}")
    print("\n")
    
    return df_awake_ambulatory

# implement the function for each participant and posture category
for (participant_id, posture), group in df_awake_ambulatory.groupby(['participant_id', 'Majority_postures']):
    df_awake_ambulatory = run_regression_and_calculate_residual(group, df_awake_ambulatory)

df_awake_ambulatory[['participant_id', 'Majority_postures', 'RSA0_msec', 'RSA0_residual', 'Calibrated_Tidal_Volume', 'Respiration_Rate_bpm']].head()


Regression Equation for Participant 12008 - Posture: Lying
RSA0_msec = 26.8618 + (5.9355 * Calibrated_Tidal_Volume) + (-0.5008 * Respiration_Rate_bpm)
P-value for Intercept: 0.0000
P-value for Calibrated_Tidal_Volume: 0.1340
P-value for Respiration_Rate_bpm: 0.0015


Regression Equation for Participant 12008 - Posture: Sitting
RSA0_msec = 33.7866 + (0.0579 * Calibrated_Tidal_Volume) + (-0.5706 * Respiration_Rate_bpm)
P-value for Intercept: 0.0000
P-value for Calibrated_Tidal_Volume: 0.9828
P-value for Respiration_Rate_bpm: 0.0000


Regression Equation for Participant 12008 - Posture: Standing
RSA0_msec = 43.3121 + (-5.0381 * Calibrated_Tidal_Volume) + (-0.6814 * Respiration_Rate_bpm)
P-value for Intercept: 0.0000
P-value for Calibrated_Tidal_Volume: 0.0063
P-value for Respiration_Rate_bpm: 0.0004


Skipping regression for Participant 13304 - Posture: Lying (Less than 20 data points)

Regression Equation for Participant 13304 - Posture: Sitting
RSA0_msec = 204.8733 + (-40.6168 * Calibra

,participant_id,Majority_postures,RSA0_msec,RSA0_residual,Calibrated_Tidal_Volume,Respiration_Rate_bpm
0,12008,Standing,12.142857,-5.358006,2.933965,16.185775
1,12008,Sitting,11.153846,-13.883491,1.706549,15.507188
2,12008,Standing,14.000000,-3.689623,2.811731,16.812479
3,12008,Standing,22.230769,2.632711,2.683670,14.958695
4,12008,Standing,19.636364,-2.231437,2.478718,13.143172


In [27]:
import statsmodels.api as sm
import numpy as np
# function to run the multiple regressions for Approach 3 and subsequently calculate the Residual RSAs - the version that first eliminates the
   #respiration rate and RSA0 values that deviate 4.5 sds or more than the mean within participant within posture
# when there are less than 20 data points for a participant's given posture category, it skips that line (not statistically apppropriate to regress then)

def run_regression_and_calculate_residual_45(group, df):
   # remove outliers ≥ 4.5 SDs from the mean for RSA0 and Respiration Rate
    rsa0_mean = group['RSA0_msec'].mean()
    rsa0_std = group['RSA0_msec'].std()
    rr_mean = group['Respiration_Rate_bpm'].mean()
    rr_std = group['Respiration_Rate_bpm'].std()

    #we already previously filtered to remove deviant calibrated tidal volume values so no need to do that again
    group_filtered = group[
        (np.abs(group['RSA0_msec'] - rsa0_mean) < 4.5 * rsa0_std) &
        (np.abs(group['Respiration_Rate_bpm'] - rr_mean) < 4.5 * rr_std)
    ]
    # checking if at least a min of 20 data points remained
    if len(group_filtered) < 20:
        print(f"Skipping regression for Participant {group['participant_id'].iloc[0]} - Posture: {group['Majority_postures'].iloc[0]} (Less than 20 data points after filtering)\n")
        df.loc[group.index, 'RSA0_residual_45'] = np.nan
        return df

    # defining the independent variables as tidal volume and respiration rate, and adding a constant
    X_filtered = sm.add_constant(group_filtered[['Calibrated_Tidal_Volume', 'Respiration_Rate_bpm']])
    # defining the dependent variable as RSA0
    y_filtered = group_filtered['RSA0_msec']
    model = sm.OLS(y_filtered, X_filtered).fit()

    # predict for *all* rows including the outliers which were not taken into account while making the regressions
    X_all = sm.add_constant(group[['Calibrated_Tidal_Volume', 'Respiration_Rate_bpm']])
    group['predicted_RSA0_45'] = model.predict(X_all)
    group['RSA0_residual_45'] = group['RSA0_msec'] - group['predicted_RSA0_45']

    # assign residuals back to full dataframe
    df.loc[group.index, 'RSA0_residual_45'] = group['RSA0_residual_45']

    intercept = model.params['const']
    tidal_volume_coef = model.params['Calibrated_Tidal_Volume']
    respiration_rate_coef = model.params['Respiration_Rate_bpm']

    intercept_pvalue = model.pvalues['const']
    tidal_volume_pvalue = model.pvalues['Calibrated_Tidal_Volume']
    respiration_rate_pvalue = model.pvalues['Respiration_Rate_bpm']

    print(f"Regression Equation for Participant {group['participant_id'].iloc[0]} - Posture: {group['Majority_postures'].iloc[0]} (Filtered for fitting only)")
    print(f"RSA0_msec = {intercept:.4f} + ({tidal_volume_coef:.4f} * Calibrated_Tidal_Volume) + ({respiration_rate_coef:.4f} * Respiration_Rate_bpm)")
    print(f"P-value for Intercept: {intercept_pvalue:.4f}")
    print(f"P-value for Calibrated_Tidal_Volume: {tidal_volume_pvalue:.4f}")
    print(f"P-value for Respiration_Rate_bpm: {respiration_rate_pvalue:.4f}\n")

    return df
    
# applying the function to the dataframe in which we group by the participant_id and posture 
for (participant_id, posture), group in df_awake_ambulatory.groupby(['participant_id', 'Majority_postures']):
    df_awake_ambulatory = run_regression_and_calculate_residual_45(group, df_awake_ambulatory)

# previewing the relevant columns
df_awake_ambulatory[['participant_id', 'Majority_postures', 'RSA0_msec', 'RSA0_residual_45', 'Calibrated_Tidal_Volume', 'Respiration_Rate_bpm']].head()


Regression Equation for Participant 12008 - Posture: Lying (Filtered for fitting only)
RSA0_msec = 26.8618 + (5.9355 * Calibrated_Tidal_Volume) + (-0.5008 * Respiration_Rate_bpm)
P-value for Intercept: 0.0000
P-value for Calibrated_Tidal_Volume: 0.1340
P-value for Respiration_Rate_bpm: 0.0015

Regression Equation for Participant 12008 - Posture: Sitting (Filtered for fitting only)
RSA0_msec = 33.0050 + (0.1294 * Calibrated_Tidal_Volume) + (-0.5380 * Respiration_Rate_bpm)
P-value for Intercept: 0.0000
P-value for Calibrated_Tidal_Volume: 0.9601
P-value for Respiration_Rate_bpm: 0.0000

Regression Equation for Participant 12008 - Posture: Standing (Filtered for fitting only)
RSA0_msec = 38.0252 + (-4.2131 * Calibrated_Tidal_Volume) + (-0.5019 * Respiration_Rate_bpm)
P-value for Intercept: 0.0000
P-value for Calibrated_Tidal_Volume: 0.0137
P-value for Respiration_Rate_bpm: 0.0058

Skipping regression for Participant 13304 - Posture: Lying (Less than 20 data points after filtering)

Regres

,participant_id,Majority_postures,RSA0_msec,RSA0_residual_45,Calibrated_Tidal_Volume,Respiration_Rate_bpm
0,12008,Standing,12.142857,-5.397732,2.933965,16.185775
1,12008,Sitting,11.153846,-13.728331,1.706549,15.507188
2,12008,Standing,14.000000,-3.741045,2.811731,16.812479
3,12008,Standing,22.230769,3.019797,2.683670,14.958695
4,12008,Standing,19.636364,-1.349283,2.478718,13.143172


In [28]:
df_awake_ambulatory[['participant_id', 'Majority_postures', 'RSA0_msec', 'RSA0_residual_45', 'Calibrated_Tidal_Volume', 'Respiration_Rate_bpm']].tail()

,participant_id,Majority_postures,RSA0_msec,RSA0_residual_45,Calibrated_Tidal_Volume,Respiration_Rate_bpm
47333,99857,Sitting,23.777778,-14.560730,1.830138,20.531699
47334,99857,Sitting,16.809524,-19.179254,1.581758,21.471635
47335,99857,Sitting,21.523810,-11.714877,1.416549,22.684089
47336,99857,Sitting,24.133333,-16.028958,2.225654,19.983675
47337,99857,Sitting,28.375000,-18.168832,2.829555,17.367665


In [36]:
df_awake_ambulatory.iloc[5000] #checking the different column and values for a random row

Unnamed: 0                                                  6733
Location                                                     NaN
Physical_Exertion                                            NaN
Posture                                                      NaN
Social_Situation                                             NaN
Type_Of_Activity                                             NaN
Location_Code                                                NaN
Physical_Exertion_Code                                       NaN
Posture_Code                                                 NaN
Social_Situation_Code                                        NaN
Type_Of_Activity_Code                                        NaN
Subject_ID                                                 25879
Label_ID                                                     398
Start_Date                                            13-04-2024
Start_Time                                          11:27:08.500
End_Date                 

In [ ]:
#for the supine (lying) condition of the participants below, a regression line could not be calculated due to a lack of at least 20 data points
#the same is the case for both RSA0_residual and RSA0_residual_45
#13304
#24340
#34714
#54783
#66303
#74998
#82556
#only 7 participants do not have a regression for the supine condition - one could argue
#we should also run the supine posture's analyses for the remaining 35 individuals. However, note that although we could statistically calculate a 
#regression line for these 35 individuals, the data points (for x or y) do not have meaningful variation with only a handful
#of data points borderline above 20 data points per individual.

In [ ]:
#continue with adding RSA0/VT, then making the ambulatory baseline regressions (and adjusted RSA), and adding the lab baseline regression based metrics

In [37]:
df_awake_ambulatory["Ttot"] = 60/df_awake_ambulatory["Respiration_Rate_bpm"] #add the breathing period column using the respiration rate data
df_awake_ambulatory["RSA0_over_Vt"] = df_awake_ambulatory['RSA0_msec']/df_awake_ambulatory['Calibrated_Tidal_Volume'] #Approach 4 (tidal volume norm RSA)

In [38]:
df_awake_ambulatory[[
    "Respiration_Rate_bpm", 
    "Calibrated_Tidal_Volume", 
    "RSA0_msec", 
    "Ttot", 
    "RSA0_over_Vt"
]].head()

,Respiration_Rate_bpm,Calibrated_Tidal_Volume,RSA0_msec,Ttot,RSA0_over_Vt
0,16.185775,2.933965,12.142857,3.706959,4.138719
1,15.507188,1.706549,11.153846,3.869173,6.535906
2,16.812479,2.811731,14.000000,3.568778,4.979138
3,14.958695,2.683670,22.230769,4.011045,8.283720
4,13.143172,2.478718,19.636364,4.565108,7.921985


In [ ]:
# make a dataframe with the Standing data of participant 12008 to double check if the br is correct when done with the function
#checked, the following cell's function gives the same result
#df_standing_12008 = df_awake_ambulatory[
 #   (df_awake_ambulatory['participant_id'] == 12008) &
 #   (df_awake_ambulatory['Majority_postures'] == 'Standing')
#]

#print(df_standing_12008)
#print(len(df_standing_12008))

#X = df_standing_12008['Ttot']
#y = df_standing_12008['RSA0_over_Vt']

# add intercept
#X = sm.add_constant(X)

# fitting the model
#model = sm.OLS(y, X).fit()

# print the model summary
#print(model.summary())

In [40]:
import statsmodels.api as sm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# this is for Approach 2: making baseline regressions (RSA/Vt vs Ttot) using ambulatory data
#here, no outlier removal is conducted (in the next cell, we will re-do this, removing outliers in the y-axis that are 4.5/more sds away from the mean)

# folder to save plots
plot_folder = 'ambulatory_br_plots'
os.makedirs(plot_folder, exist_ok=True)

#defining list to store the regression coefficients
ambulatory_br_list = []

# function to make the ambulatory baseline regression (br)
def run_ambulatory_br_regression(group):
    if len(group) < 20: #check if at least 20 points are available
        print(f"Skipping regression for Participant {group['participant_id'].iloc[0]} - Posture: {group['Majority_postures'].iloc[0]} (Less than 20 points)\n")
        return

    X = sm.add_constant(group[['Ttot']]) #x variable and adding a constant to it
    y = group['RSA0_over_Vt'] #y axis

    model = sm.OLS(y, X).fit() #fitting the model for the participant-posture combination

    intercept = model.params['const']
    slope = model.params['Ttot']
    intercept_pval = model.pvalues['const']
    slope_pval = model.pvalues['Ttot']

    print(f"Regression for Participant {group['participant_id'].iloc[0]} - Posture: {group['Majority_postures'].iloc[0]}")
    print(f"RSA0_over_Vt = {intercept:.4f} + ({slope:.4f} * Ttot)")
    print(f"P-value for Intercept: {intercept_pval:.4f}")
    print(f"P-value for Ttot: {slope_pval:.4f}\n")

    # saving the participant_id, posture, as well as the associated intercept value, slope value, number of points used, and the p values to the
    #pre-defined list
    ambulatory_br_list.append({
        'participant_id': group['participant_id'].iloc[0],
        'posture': group['Majority_postures'].iloc[0],
        'intercept': intercept,
        'slope': slope,
        'intercept_pval': intercept_pval,
        'slope_pval': slope_pval,
        'n_points': len(group)
    })

    # plot
    plt.figure(figsize=(6, 4))
    plt.scatter(group['Ttot'], group['RSA0_over_Vt'], alpha=0.6, label='Data Points')
    x_vals = np.linspace(group['Ttot'].min(), group['Ttot'].max(), 100)
    y_vals = intercept + slope * x_vals
    plt.plot(x_vals, y_vals, color='red', label='Regression Line')
    plt.xlabel('Ttot')
    plt.ylabel('RSA0_over_Vt')
    plt.title(f"Participant {group['participant_id'].iloc[0]} - {group['Majority_postures'].iloc[0]}")
    plt.legend()
    plt.tight_layout()

    # saving the generated plot as a file and save under the folder
    filename = f"{group['participant_id'].iloc[0]}_{group['Majority_postures'].iloc[0].replace(' ', '_')}.png"
    plt.savefig(os.path.join(plot_folder, filename))
    plt.close()

# looping over all participant-posture groups
for (participant_id, posture), group in df_awake_ambulatory.groupby(['participant_id', 'Majority_postures']):
    run_ambulatory_br_regression(group)

# convert the list into a dataframe
ambulatory_br_df = pd.DataFrame(ambulatory_br_list)
ambulatory_br_df.head()

#save the coefficients as an excel
ambulatory_br_df.to_excel("ambulatory_br_df.xlsx")

Regression for Participant 12008 - Posture: Lying
RSA0_over_Vt = 38.7635 + (-2.6556 * Ttot)
P-value for Intercept: 0.0000
P-value for Ttot: 0.1137

Regression for Participant 12008 - Posture: Sitting
RSA0_over_Vt = 14.4582 + (1.0403 * Ttot)
P-value for Intercept: 0.0000
P-value for Ttot: 0.0003

Regression for Participant 12008 - Posture: Standing
RSA0_over_Vt = -0.6066 + (1.7568 * Ttot)
P-value for Intercept: 0.5513
P-value for Ttot: 0.0000

Skipping regression for Participant 13304 - Posture: Lying (Less than 20 points)

Regression for Participant 13304 - Posture: Sitting
RSA0_over_Vt = 398.0578 + (-48.7893 * Ttot)
P-value for Intercept: 0.0000
P-value for Ttot: 0.0048

Regression for Participant 13304 - Posture: Standing
RSA0_over_Vt = 13.0335 + (12.6931 * Ttot)
P-value for Intercept: 0.6252
P-value for Ttot: 0.1260

Regression for Participant 15337 - Posture: Lying
RSA0_over_Vt = 242.1322 + (-30.4039 * Ttot)
P-value for Intercept: 0.0180
P-value for Ttot: 0.2479

Regression for Par

In [41]:
# this is for Approach 2: making baseline regressions (RSA/Vt vs Ttot) using ambulatory data
#here, outlier removal is conducted, we are removing outliers in the y-axis that are 4.5/more sds away from the mean, after this check if there are
#at least 20 points remaining

import statsmodels.api as sm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# folder to save the RSA0/Vt vs Ttot plots that will be generated following outlier removal over the y axis
plot_folder_45 = 'ambulatory_br_plots_4_5sd'
os.makedirs(plot_folder_45, exist_ok=True)

# to store the regression coefficients
ambulatory_br_45_list = []

# function to get the ambulatory baseline regressions following outlier removal
def run_ambulatory_br_45(group):
    # for each "group" (i.e., the group of data for a given posture for a given participant)
    mean_val = group['RSA0_over_Vt'].mean()
    std_val = group['RSA0_over_Vt'].std()
    group_filtered = group[np.abs(group['RSA0_over_Vt'] - mean_val) < 4.5 * std_val]

    if len(group_filtered) < 20:  #after the filtering, check whether a minimum of 20 points are still remaining for the "group"
        print(f"Skipping regression for Participant {group['participant_id'].iloc[0]} - Posture: {group['Majority_postures'].iloc[0]} (Less than 20 points after 4.5 SD filtering)\n")
        return

    X = sm.add_constant(group_filtered[['Ttot']])
    y = group_filtered['RSA0_over_Vt']
    model = sm.OLS(y, X).fit() #fitting the model

    intercept = model.params['const']
    slope = model.params['Ttot']
    intercept_pval = model.pvalues['const']
    slope_pval = model.pvalues['Ttot']

    print(f"Filtered (4.5 SD) Regression for Participant {group['participant_id'].iloc[0]} - Posture: {group['Majority_postures'].iloc[0]}")
    print(f"RSA0_over_Vt = {intercept:.4f} + ({slope:.4f} * Ttot)")
    print(f"P-value for Intercept: {intercept_pval:.4f}")
    print(f"P-value for Ttot: {slope_pval:.4f}\n")

    ambulatory_br_45_list.append({
        'participant_id': group['participant_id'].iloc[0],
        'posture': group['Majority_postures'].iloc[0],
        'intercept': intercept,
        'slope': slope,
        'intercept_pval': intercept_pval,
        'slope_pval': slope_pval,
        'n_points': len(group_filtered)
    })

    # plot
    plt.figure(figsize=(6, 4))
    plt.scatter(group_filtered['Ttot'], group_filtered['RSA0_over_Vt'], alpha=0.6, label='Filtered Data')
    x_vals = np.linspace(group_filtered['Ttot'].min(), group_filtered['Ttot'].max(), 100)
    y_vals = intercept + slope * x_vals
    plt.plot(x_vals, y_vals, color='red', label='Regression Line')
    plt.xlabel('Ttot')
    plt.ylabel('RSA0_over_Vt')
    plt.title(f"Participant {group['participant_id'].iloc[0]} - {group['Majority_postures'].iloc[0]} (Filtered 4.5 SD)")
    plt.legend()
    plt.tight_layout()

    filename = f"{group['participant_id'].iloc[0]}_{group['Majority_postures'].iloc[0].replace(' ', '_')}_filtered.png"
    plt.savefig(os.path.join(plot_folder_45, filename))
    plt.close()

# looping over each "group" (i.e., participant-posture combination) and implementing the function to do outlier removall, calculating br, plotting
for (participant_id, posture), group in df_awake_ambulatory.groupby(['participant_id', 'Majority_postures']):
    run_ambulatory_br_45(group)

# convert the list with the participant id, posture, regression coefficients into a dataframe
ambulatory_br_45 = pd.DataFrame(ambulatory_br_45_list)

ambulatory_br_45.head()

#save the coefficients as an excel
ambulatory_br_45.to_excel("ambulatory_br_45.xlsx")

Filtered (4.5 SD) Regression for Participant 12008 - Posture: Lying
RSA0_over_Vt = 38.7635 + (-2.6556 * Ttot)
P-value for Intercept: 0.0000
P-value for Ttot: 0.1137

Filtered (4.5 SD) Regression for Participant 12008 - Posture: Sitting
RSA0_over_Vt = 14.6187 + (0.9817 * Ttot)
P-value for Intercept: 0.0000
P-value for Ttot: 0.0004

Filtered (4.5 SD) Regression for Participant 12008 - Posture: Standing
RSA0_over_Vt = 0.7422 + (1.4003 * Ttot)
P-value for Intercept: 0.4296
P-value for Ttot: 0.0000

Skipping regression for Participant 13304 - Posture: Lying (Less than 20 points after 4.5 SD filtering)

Filtered (4.5 SD) Regression for Participant 13304 - Posture: Sitting
RSA0_over_Vt = 398.0578 + (-48.7893 * Ttot)
P-value for Intercept: 0.0000
P-value for Ttot: 0.0048

Filtered (4.5 SD) Regression for Participant 13304 - Posture: Standing
RSA0_over_Vt = 13.0335 + (12.6931 * Ttot)
P-value for Intercept: 0.6252
P-value for Ttot: 0.1260

Filtered (4.5 SD) Regression for Participant 15337 - Pos

In [42]:
#defining the function to calculate the delta RSA0/VT values using the ambulatory baseline regressions - Approach 2
#VERY IMPORTANT VARIABLE NAME CLARIFICATION: You will see that we named the ambulatory baseline regression cardiac vagal activity as
#delta_RSA0_over_Ttot_ambulatory or delta_RSA0_over_Ttot_ambulatory_45. This variable, as also explained and calculated below is
#delta RSA0/Vt. We simply named it as delta_RSA0_over_Ttot to avoid making mistakes in code as the different RSA variables have similar names and terms

def apply_residuals(df, regression_df, column_name):#df would be the actual physiological data, regression_df is the dataframe with ambulatory df values
    residuals = []                                   #column name is the new column that would be added to df for the newly calculated deltaRSA0/Vt

    for idx, row in df.iterrows():
        match = regression_df[ #match is selecting the row that is the same as the current df row's participant id and posture
            (regression_df['participant_id'] == row['participant_id']) &  # for the current row in the df, searching for the 
                                                                            # regression of that participant and posture
            (regression_df['posture'] == row['Majority_postures'])
        ]
        if not match.empty:
            intercept = match.iloc[0]['intercept']
            slope = match.iloc[0]['slope']
            predicted = intercept + slope * row['Ttot']   #calculate the predicted RSA0/Vt given the row's Ttot
            residual = row['RSA0_over_Vt'] - predicted   #subtract the observed from the predicted to get at the delta
        else:
            residual = np.nan
            
        residuals.append(residual)

    df[column_name] = residuals
    return df

# using the ambulatory baseline regressions, calculating deltaRSA0/Vt
df_awake_ambulatory = apply_residuals(df_awake_ambulatory, ambulatory_br_df, 'delta_RSA0_over_Ttot_ambulatory') #again, this is deltaRSA0/Vt

# using the ambulatory baseline regressions (while making the br points at or away 4.5sds from the mean were removed), calculating deltaRSA0/Vt
df_awake_ambulatory = apply_residuals(df_awake_ambulatory, ambulatory_br_45, 'delta_RSA0_over_Ttot_ambulatory_45') #again, this is deltaRSA0/Vt


In [33]:
#save to also manually check if the calculations worked
#df_awake_ambulatory.to_excel("df_after_ambulatory_approach2.xlsx")   
#manual check verified that it does work

In [43]:
#and now, calculating the deltaRSA0/Vt for Approach 1, using the laboratory baseline regressions calculated with the paced breathing data
import os
import pandas as pd
import numpy as np

# the directory where the laboratory calibration data is stored
base_path = r"C:\Users\msa583\OneDrive - Vrije Universiteit Amsterdam\Desktop\RSA Analyses Post-Processing\RQ2"

# mapping the postures in the dataframe to the names used in the lab calibration excels
posture_mapping = {
    'lying': 'supine',
    'sitting': 'sitting',
    'standing': 'standing'
}

# function for reading in the lab calibration files of participants and turning into pandas dataframe
def get_lab_coefficients(participant_id):
    file_path = os.path.join(base_path, f"lab_calibration_{participant_id}.xlsx") #this is how the individual files are titled
    if os.path.exists(file_path):
        return pd.read_excel(file_path) #making it a pandas dataframe
    else:
        print(f"File not found for Participant {participant_id}")
        return None

# function to calculate the deltaRSA0/Vt
def calculate_lab_delta_cached(df, suffix, column_name):
    deltas = []
    lab_cache = {}  # dictionary to store each participant's lab calibration dataframe (so we don't need to re-read the excel file for every row)

    for idx, row in df.iterrows():
        ppid = row['participant_id'] #extracting the participant id of the current row
        posture_raw = row['Majority_postures'].strip().lower() #extracting the posture of the current row
        posture = posture_mapping.get(posture_raw) #translate the posture name (relevant if converting from lying to supine)

        if posture is None:
            print(f"Unknown posture '{posture_raw}' for participant {ppid}")
            deltas.append(np.nan)
            continue

        # using the dictionary to avoid re-reading excel files for every row
        if ppid not in lab_cache:
            lab_cache[ppid] = get_lab_coefficients(ppid)

        lab_df = lab_cache[ppid]

        if lab_df is None:
            deltas.append(np.nan)
            continue

        try:   #the regressions are already calculated previously in the separate participant specific notebooks which is why we do not fit a model here
            slope_col = f'br_slope_{posture}{suffix}'
            intercept_col = f'br_intercept_{posture}{suffix}'

            slope = lab_df.at[0, slope_col]
            intercept = lab_df.at[0, intercept_col]

            predicted = intercept + slope * row['Ttot']
            delta = row['RSA0_over_Vt'] - predicted
            deltas.append(delta)
        except KeyError:
            print(f"Missing regression columns for posture '{posture}' (original: '{posture_raw}') in Participant {ppid}'s file")
            deltas.append(np.nan)

    df[column_name] = deltas
    return df

# apply the function to get the deltaRSA0/Vt
df_awake_ambulatory = calculate_lab_delta_cached(df_awake_ambulatory, '', 'delta_RSA0_over_Ttot_laboratory') #deltaRSA0/Vt calculated using lab br
df_awake_ambulatory = calculate_lab_delta_cached(df_awake_ambulatory, '_45', 'delta_RSA0_over_Ttot_laboratory_45') #calculated using lab br except for 4.5 SD deviant points
df_awake_ambulatory = calculate_lab_delta_cached(df_awake_ambulatory, '_3', 'delta_RSA0_over_Ttot_laboratory_3') #calculated using lab br except for 3 SD deviant points

In [44]:
# loading the descriptives (age, sex, BMI, smoking status, IPAQ will be used later in multilevel modeling)
descriptives_path = os.path.join(base_path, "Descriptives.xlsx")
df_descriptives = pd.read_excel(descriptives_path)

# merging the descriptives column onto df_awake_ambulatory based on the participant_id column
df_awake_ambulatory = df_awake_ambulatory.merge(df_descriptives, on='participant_id', how='left')

In [45]:
df_awake_ambulatory.to_excel("df_RSAmetrics_descriptives.xlsx") #the final dataframe with all the RSA metrics and descriptives, has all postures

In [46]:
df_awake_ambulatory = pd.read_excel("df_RSAmetrics_descriptives.xlsx")

In [47]:
# separating the main dataframe into three based on posture
final_df_lying = df_awake_ambulatory[df_awake_ambulatory['Majority_postures'].str.lower() == 'lying'].copy()
final_df_sitting = df_awake_ambulatory[df_awake_ambulatory['Majority_postures'].str.lower() == 'sitting'].copy()
final_df_standing = df_awake_ambulatory[df_awake_ambulatory['Majority_postures'].str.lower() == 'standing'].copy()

In [50]:
final_df_lying.columns.tolist()

['Unnamed: 0.1',
 'Unnamed: 0',
 'Location',
 'Physical_Exertion',
 'Posture',
 'Social_Situation',
 'Type_Of_Activity',
 'Location_Code',
 'Physical_Exertion_Code',
 'Posture_Code',
 'Social_Situation_Code',
 'Type_Of_Activity_Code',
 'Subject_ID',
 'Label_ID',
 'Start_Date',
 'Start_Time',
 'End_Date',
 'End_Time',
 'Label_Duration_s',
 'Total_Motility_mg',
 'Average_IBI_msec',
 'Average_HR_bpm',
 'SDNN_msec',
 'Respiration_Rate_bpm',
 'Tidal_Volume_mOhm',
 'RMSSD_msec',
 'LF_ms',
 'RSA0_msec',
 'HF_ms',
 'PEP_msec',
 'LVET_msec',
 'TWave_amplitude_mV',
 'Stroke_Volume_(Nederend_2017)_cc',
 'Minute_Volume_(Nederend_2017)_lmin',
 'Average_SCL_uS',
 'nsSCRs_per_minute_ppm',
 'Min_IBI_msec',
 'Max_IBI_msec',
 'RSA_msec',
 'Artefact_Free_ECG_percent',
 'Artefact_Free_Respiration_percent',
 'Majority_postures',
 'participant_id',
 'day',
 'Experimental_Conditions',
 'Experimental_Conditions_Code',
 'Artefact_Free_ECG_s',
 'Calibrated_Tidal_Volume',
 'datetime',
 'Cleaned_Tidal_Volume',
 '

# SEPARATING THE FINAL DATAFRAME FOR RQ2 INTO LYING, SITTING, AND STANDING DATAFRAMES FOR USE IN R

In [51]:
#save as excel files, from there on move to R
final_df_lying.to_excel("final_df_lying.xlsx", index=False)
final_df_sitting.to_excel("final_df_sitting.xlsx", index=False)
final_df_standing.to_excel("final_df_standing.xlsx", index=False)

# READING IN AND MERGING THE M-PATH (EMA) FILES, CALCULATING THE TOTAL MOMENTARY PERCEIVED STRESS, POSITIVE AFFECT, NEGATIVE AFFECT SCORES

In [58]:
import pandas as pd
import glob
import os

# defining the path where the ema excels are
base_path = r"C:\Users\msa583\OneDrive - Vrije Universiteit Amsterdam\Desktop\RSA Analyses Post-Processing\RQ2"

# finding the excel files that start with m-Path_
file_list = glob.glob(os.path.join(base_path, "m-Path_*.xlsx"))

# list to append the m_Path data
df_list = []

# loop through the files
for file_path in file_list:
    filename = os.path.basename(file_path)
    participant_id = filename.split("_")[1].split(".")[0]  # extract the participant id from the file name
    
    df = pd.read_excel(file_path)
    df['participant_id'] = participant_id  # add participant ID
    
    df_list.append(df)

# converting from a list to a concetenated dataframe
df_ema = pd.concat(df_list, ignore_index=True)

#commented the printout of the dataframe to conceal the sensitive daily life emotion data
#df_ema

In [59]:
df_ema["Tot_Stress"] = df_ema["PSS_1 (sliderNegPos)"] + df_ema["PSS_2 (sliderNegPos)"] + df_ema["PSS_3 (sliderNegPos)"] + df_ema["PSS_4 (sliderNegPos)"]
df_ema["Safety"] = df_ema["safety (sliderNegPos)"]
#df_ema

In [60]:
# "affect_1 (sliderNegPos)" is happy
# "affect_2 (sliderNegPos)" is energetic
# "affect_3 (sliderNegPos)" is satisfied
# "affect_4 (sliderNegPos)" is relaxed
# "affect_5 (sliderNegPos)" is stressed
# "affect_6 (sliderNegPos)" is anxious
# "affect_7 (sliderNegPos)" is irritated
# "affect_8 (sliderNegPos)" is down

In [61]:
df_ema["Pos_Aff"] = df_ema["affect_1 (sliderNegPos)"] + df_ema["affect_2 (sliderNegPos)"] + df_ema["affect_3 (sliderNegPos)"] + df_ema["affect_4 (sliderNegPos)"]
df_ema["Neg_Aff"] = df_ema["affect_5 (sliderNegPos)"] + df_ema["affect_6 (sliderNegPos)"] + df_ema["affect_7 (sliderNegPos)"] + df_ema["affect_8 (sliderNegPos)"] 
#df_ema

In [62]:
# convert the 'Date and time' column to datetime with the to_datetime function of pandas
df_ema['Date and time'] = pd.to_datetime(df_ema['Date and time'], errors='coerce')
#print(df_ema.head())

# set it as the index
df_ema.set_index('Date and time', inplace=True)

#df_ema

In [64]:
#some participants had their start times for some VU-DAMS files have one hour or one day before/after the recording actually started
#these instances were clearly outlined as the procedures always started around 10:15 am, and the day1 and day2 recording days were always back to back
#we also had additional pieces of information such as the manually written down end time of the procedure, and the m-Path datetimes
#here we shortly go back to the df_awake_ambulatory dataset (will continue working on the df_ema and merging these in the later cells)
from pandas.tseries.frequencies import to_offset

# shifting the day1 epochs for participant 91169 1 hour backwards (earlier)
df_awake_ambulatory.loc[
    (df_awake_ambulatory['participant_id'] == 91169) & 
    (df_awake_ambulatory['day'] == 1), 
    'datetime'
] -= pd.Timedelta(hours=1)

# shifting the day2 epochs for participant 37092 1 day backwards (earlier)
df_awake_ambulatory.loc[
    (df_awake_ambulatory['participant_id'] == 37092) & 
    (df_awake_ambulatory['day'] == 2), 
    'datetime'
] -= pd.Timedelta(days=1)

# shifting epochs 1 hour forwards (later) for both day1 and day2 for the required participants
forward_shift_ids = [46773, 30739, 93676, 37818, 12008, 25201, 88137]

df_awake_ambulatory.loc[
    df_awake_ambulatory['participant_id'].isin(forward_shift_ids) & 
    df_awake_ambulatory['day'].isin([1, 2]), 
    'datetime'
] += pd.Timedelta(hours=1)


In [65]:
df_awake_ambulatory[df_awake_ambulatory['participant_id'] == 91169][['day', 'datetime']].head()
#checking a participant to see if it adjusted correctly, note that we are viewing the part of the dataframe that already excluded the lab portion
#so we see the epochs starting from the end of the procedure, which should indeed start around 11:55:00 --> this is correctly adjusted

,day,datetime
40839,1,2024-04-30 11:55:53.500
40840,1,2024-04-30 11:56:53.500
40841,1,2024-04-30 11:57:53.500
40842,1,2024-04-30 11:58:53.500
40843,1,2024-04-30 11:59:53.500


In [66]:
df_awake_ambulatory[
    (df_awake_ambulatory['participant_id'] == 37092) & 
    (df_awake_ambulatory['day'] == 2)
][['day', 'datetime']].head()
#--> correctly adjusted

,day,datetime
13465,2,2024-05-31 07:52:49.500
13466,2,2024-05-31 07:53:49.500
13467,2,2024-05-31 07:54:49.500
13468,2,2024-05-31 07:55:49.500
13469,2,2024-05-31 07:56:49.500


In [ ]:
#save to also manually check if the calculations worked
df_awake_ambulatory.to_excel("df_withRSA_withDescriptives_shifted.xlsx")   #shifted refers to the fact that we adjusted for some participants
   #their initially off-time (e.g., those who needed to get their day1/day2 files shifted by an hour/day forward/backward)
   #this does not yet have the EMA

In [80]:
df_awake_ambulatory.columns.tolist()

['Unnamed: 0.1',
 'Unnamed: 0',
 'Location',
 'Physical_Exertion',
 'Posture',
 'Social_Situation',
 'Type_Of_Activity',
 'Location_Code',
 'Physical_Exertion_Code',
 'Posture_Code',
 'Social_Situation_Code',
 'Type_Of_Activity_Code',
 'Subject_ID',
 'Label_ID',
 'Start_Date',
 'Start_Time',
 'End_Date',
 'End_Time',
 'Label_Duration_s',
 'Total_Motility_mg',
 'Average_IBI_msec',
 'Average_HR_bpm',
 'SDNN_msec',
 'Respiration_Rate_bpm',
 'Tidal_Volume_mOhm',
 'RMSSD_msec',
 'LF_ms',
 'RSA0_msec',
 'HF_ms',
 'PEP_msec',
 'LVET_msec',
 'TWave_amplitude_mV',
 'Stroke_Volume_(Nederend_2017)_cc',
 'Minute_Volume_(Nederend_2017)_lmin',
 'Average_SCL_uS',
 'nsSCRs_per_minute_ppm',
 'Min_IBI_msec',
 'Max_IBI_msec',
 'RSA_msec',
 'Artefact_Free_ECG_percent',
 'Artefact_Free_Respiration_percent',
 'Majority_postures',
 'participant_id',
 'day',
 'Experimental_Conditions',
 'Experimental_Conditions_Code',
 'Artefact_Free_ECG_s',
 'Calibrated_Tidal_Volume',
 'datetime',
 'Cleaned_Tidal_Volume',
 '

In [67]:
import pandas as pd
import numpy as np
# identifying which columns are numerical via np.number and selecting them, while excluding participant id and day 
numeric_cols = df_awake_ambulatory.select_dtypes(include=[np.number]).columns.difference(['participant_id', 'day'])

averaged_physio_data = [] # empty list to store the physiological data
num_epochs_list = []  #empty list to store the number of epochs averaged to reach the 5min averages (it won't always be 5 epochs)
majority_posture_list = [] #empty list to store the majority posture for the 5 min epoch

# iterrows iterates through each row in the dataframe df_ema
for idx, row in df_ema.iterrows():
    pid = int(row['participant_id']) #extract the participant id of the current row
    #we previously set a datetime index to the dataframe df_ema so the time is now in the index (time is no longer in the column)
    ema_time = row.name  # 'Date and time' is the index, so the row.name gets the time when the prompt is answered
    
    # filter the physiological data (from df_awake_ambulatory) between 5 minutes before the EMA prompt and the EMA prompt, for the current participant
    physio_window = df_awake_ambulatory[
        (df_awake_ambulatory['participant_id'] == pid) &
        (df_awake_ambulatory['datetime'] >= ema_time - pd.Timedelta(minutes=5)) &
        (df_awake_ambulatory['datetime'] < ema_time)   #backward-looking 5 minute window
    ]
    
    if not physio_window.empty:  #if this window is not empty (if there is any data)
        averaged_vals = physio_window[numeric_cols].mean()   #calculate the mean for all numerical columns (e.g., RSA0_msec and so on) for this df subset
                                                   #the function .mean on a dataframes operates columnwise by default
        epoch_count = len(physio_window) #the number of epochs in the window
        majority_posture = physio_window['Majority_postures'].mode() #the majority posture is the most frequently occuring posture across epochs
        majority_posture = majority_posture.iloc[0] if not majority_posture.empty else np.nan #if there is a tie, iloc[0] returns the first in the list
    else: #if there is no data in the window
        averaged_vals = pd.Series([np.nan] * len(numeric_cols), index=numeric_cols) #put in NaNs for the numeric columns
        epoch_count = 0  #put in 0 for the number of epochs
        majority_posture = np.nan  #put in NaN for the posture

    averaged_physio_data.append(averaged_vals)
    num_epochs_list.append(epoch_count)
    majority_posture_list.append(majority_posture)

# combine into a dataframe
averaged_df = pd.DataFrame(averaged_physio_data)
averaged_df['num_epochs_averaged'] = num_epochs_list
averaged_df['Majority_posture_5min'] = majority_posture_list

# concatenate the averaged_df with df_ema
df_ema_merged = pd.concat([df_ema.reset_index(), averaged_df], axis=1)
df_ema_merged.set_index('Date and time', inplace=True)  

In [1]:
#averaged_df

In [68]:
#the viewing of these ema dataframes are purposefully commented out so the sensitive emotional state data is not be visible on GitHub
#df_ema_merged

In [69]:
len(df_ema_merged)

874

In [71]:
#df_ema_merged["RSA0_msec"]

In [72]:
#drop the rows where no physiological epoch could be found - this would be due to EMA data being answered when the wearable is not worn (e.g., charging)
df_ema_merged = df_ema_merged.dropna(subset=['RSA0_msec'])
#df_ema_merged

In [73]:
len(df_ema_merged)

735

In [75]:
df_ema_merged["Calibrated_Tidal_Volume"]

Date and time
2024-09-02 12:33:26    2.942938
2024-09-02 13:41:26    2.951459
2024-09-02 13:43:25    3.071775
2024-09-02 14:51:48    1.841011
2024-09-02 15:31:20    1.260392
2024-09-02 16:27:28    2.962626
2024-09-02 17:38:32    1.642151
2024-09-02 19:08:06    2.728619
2024-09-03 09:49:05    1.417763
2024-09-03 10:43:35    2.783556
2024-09-03 11:01:59    1.211268
2024-09-03 12:21:23    1.363699
2024-09-03 13:41:40    2.687845
2024-09-03 14:15:02    2.669923
2024-09-03 15:31:37    0.563748
2024-09-03 16:06:52    1.238459
2024-09-03 17:01:49    1.212279
2024-09-03 18:49:39    1.711617
2024-09-03 20:08:05    0.524553
2024-09-03 21:27:19    2.318793
2024-08-09 13:29:16    0.674715
2024-08-09 14:15:31    0.268139
2024-08-09 15:44:03    0.898143
2024-06-25 12:37:02    0.831461
2024-06-25 14:00:27    0.797135
2024-06-25 14:16:44    0.777925
2024-06-25 16:04:13    0.706917
2024-06-25 17:47:32    0.903963
2024-06-25 18:25:38    0.682636
2024-06-25 19:14:39    0.495974
2024-06-25 21:12:15    0.8

In [76]:
df_ema_merged['Majority_posture_5min'].value_counts(normalize=True) * 100

Majority_posture_5min
Sitting     66.530612
Standing    25.034014
Lying        8.435374
Name: proportion, dtype: float64

In [81]:
df_ema_merged.columns.tolist()

['PSS_1 (sliderNegPos)',
 'PSS_2 (sliderNegPos)',
 'PSS_3 (sliderNegPos)',
 'PSS_4 (sliderNegPos)',
 'safety (sliderNegPos)',
 'location (multipleChoice)',
 'with_who (multipleChoice)',
 'activity_type (multipleChoice)',
 'panas_0 (multipleChoice)',
 'affect_1 (sliderNegPos)',
 'affect_2 (sliderNegPos)',
 'affect_3 (sliderNegPos)',
 'affect_4 (sliderNegPos)',
 'affect_5 (sliderNegPos)',
 'affect_6 (sliderNegPos)',
 'affect_7 (sliderNegPos)',
 'affect_8 (sliderNegPos)',
 'language (multipleChoice)',
 'Self_Speech (sliderNegPos)',
 'consumption (multipleChoice)',
 'system_devspecs (devicespecs)',
 'participant_id',
 'Tot_Stress',
 'Safety',
 'Pos_Aff',
 'Neg_Aff',
 'Age',
 'Artefact_Free_ECG_percent',
 'Artefact_Free_ECG_s',
 'Artefact_Free_Respiration_percent',
 'Average_HR_bpm',
 'Average_IBI_msec',
 'Average_SCL_uS',
 'BMI',
 'Calibrated_Tidal_Volume',
 'Cleaned_Tidal_Volume',
 'Current_Smoking',
 'Experimental_Conditions',
 'Experimental_Conditions_Code',
 'HF_ms',
 'Height',
 'IPAQ_

In [82]:
import pandas as pd
#Sex is missing in the df_ema_merged so add that
descriptives_df = pd.read_excel("Descriptives.xlsx")

df_ema_merged['participant_id'] = df_ema_merged['participant_id'].astype(str)
descriptives_df['participant_id'] = descriptives_df['participant_id'].astype(str)

df_ema_merged = pd.merge(df_ema_merged, descriptives_df[['participant_id', 'Sex']], on='participant_id', how='left')  #only keeping the id/sex columns from descriptived
#print(df_ema_merged.head())

In [83]:
df_ema_merged.columns

Index(['PSS_1 (sliderNegPos)', 'PSS_2 (sliderNegPos)', 'PSS_3 (sliderNegPos)',
       'PSS_4 (sliderNegPos)', 'safety (sliderNegPos)',
       'location (multipleChoice)', 'with_who (multipleChoice)',
       'activity_type (multipleChoice)', 'panas_0 (multipleChoice)',
       'affect_1 (sliderNegPos)', 'affect_2 (sliderNegPos)',
       'affect_3 (sliderNegPos)', 'affect_4 (sliderNegPos)',
       'affect_5 (sliderNegPos)', 'affect_6 (sliderNegPos)',
       'affect_7 (sliderNegPos)', 'affect_8 (sliderNegPos)',
       'language (multipleChoice)', 'Self_Speech (sliderNegPos)',
       'consumption (multipleChoice)', 'system_devspecs (devicespecs)',
       'participant_id', 'Tot_Stress', 'Safety', 'Pos_Aff', 'Neg_Aff', 'Age',
       'Artefact_Free_ECG_percent', 'Artefact_Free_ECG_s',
       'Artefact_Free_Respiration_percent', 'Average_HR_bpm',
       'Average_IBI_msec', 'Average_SCL_uS', 'BMI', 'Calibrated_Tidal_Volume',
       'Cleaned_Tidal_Volume', 'Current_Smoking', 'Experimental_Conditi

In [84]:
df_ema_merged.to_excel("df_ema_merged.xlsx")    #refers to the EMA prompts of all participants along with the averaged physiological and posture data

# Calculating Descriptive Summary

In [86]:
# loading the descriptives (age, sex, BMI, smoking status, IPAQ will be used later in multilevel modeling)
import pandas as pd
df_descriptives = pd.read_excel("Descriptives.xlsx")

In [87]:
#df_descriptives

In [88]:
# defining special mappings for numeric-coded categorical variables
special_categoricals = {
    'Current_Smoking': {0: 'Non-smoking', 1: 'Smoking'},
    'IPAQ_SF': {1: 'Minimally active', 2: 'Moderately active', 3: 'HEPA active'}
}

# apply the mappings
for col, mapping in special_categoricals.items():
    if col in df_descriptives.columns:
        df_descriptives[col] = df_descriptives[col].map(mapping).astype("category")

# get list of categorical columns (including object types and our recoded ones)
categorical_cols = df_descriptives.select_dtypes(include=["object", "category"]).columns

# loop through and print summaries
for col in df_descriptives.columns:
    print(f"\n--- {col} ---")
    
    if col in categorical_cols:
        value_counts = df_descriptives[col].value_counts(dropna=False, normalize=True) * 100
        for category, percent in value_counts.items():
            print(f"{category}: {percent:.2f}%")
    elif pd.api.types.is_numeric_dtype(df_descriptives[col]):
        col_data = df_descriptives[col].dropna()
        print(f"Mean: {col_data.mean():.2f}")
        print(f"SD: {col_data.std():.2f}")
        print(f"Min: {col_data.min():.2f}")
        print(f"Max: {col_data.max():.2f}")



--- participant_id ---
Mean: 55858.50
SD: 25760.66
Min: 12008.00
Max: 99857.00

--- Height ---
Mean: 167.25
SD: 6.96
Min: 154.00
Max: 187.80

--- Weight ---
Mean: 66.51
SD: 12.45
Min: 49.30
Max: 103.40

--- Sex ---
Female: 85.71%
Male: 14.29%

--- Ethnicity ---
White/Caucasian: 66.67%
West-Asian/ Middle East / North Africa: 16.67%
Mixed race/ethnicity, namely:: 4.76%
South-east asia: 4.76%
Hispanic/Latino: 2.38%
Black (African black, Afro-American,  Afro-Caribbean, African American): 2.38%
East-asian (China, Japan, Taiwan, North- and South Korea): 2.38%

--- Age ---
Mean: 23.81
SD: 6.99
Min: 18.00
Max: 47.00

--- BMI ---
Mean: 23.73
SD: 3.83
Min: 19.30
Max: 36.70

--- BMI_Category ---
Healthy Weight: 78.57%
Overweight: 9.52%
Obesity: 9.52%
Severe Obesity: 2.38%

--- Education ---
Some University coursework completed but no degree (HBO, WO): 38.10%
Completed Secondary School: 33.33%
Master's level degree (MA, MS, MBA): 11.90%
University Bachelors Degree (HBO, WO Bachelor): 11.90%
Docto